# Stage 3 — Data Warehouse

This notebook loads the Stage 2 output (`data/processed/cleaned_jobs.csv`) into a **SQLite star schema** at `data/processed/skillmap.db`. It then runs three OLAP queries:
1. Top 10 most demanded skills
2. Average salary by experience level
3. Job count by industry and location

Company, posting-date, and industry-ID attributes that `cleaned_jobs.csv` doesn't carry are read from the raw LinkedIn Job Postings tables. The logic lives in `src/warehouse.py`. To run without the notebook: `python -m src.warehouse`.

## Step 1 — Setup

In [ ]:
import logging
import sqlite3
import sys
from pathlib import Path

import pandas as pd


def find_root(start: Path) -> Path:
    """Return the first directory at or above `start` that contains CLAUDE.md."""
    for candidate in [start, *start.parents]:
        if (candidate / "CLAUDE.md").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root (no CLAUDE.md found above cwd).")


PROJECT_ROOT = find_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT))

from src import warehouse as wh

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
OUTPUT_DIR = PROJECT_ROOT / "outputs"

## Step 2 — Schema design

```
                     dim_time            dim_skills
                    (time_id)            (skill_id, skill_name, skill_type)
                         \                 /
 dim_location ──── job_postings (FACT) ────  dim_company
 (location_id,     job_id, skill_id,         (company_id, company_name,
  city, state)     location_id, company_id,   company_size, employee_count)
                   time_id, salary_tier,
                   experience_level,
                   normalized_salary
                         │ job_id
               bridge_job_industry ── dim_industry
```

- **Fact grain: one row per (job, skill).** This follows the specified fact table, which includes `skill_id`. Both skill types are loaded: `skill_type = 'extracted'` (the 100-skill vocabulary matched in descriptions) and `'category'` (the 35 LinkedIn job-function categories). A job with no skills gets one row with `skill_id` NULL, so no job is lost.
- **`v_jobs` view: one row per job.** Salary is repeated on each of a job's skill rows, so averaging over the fact table would weight jobs by how many skills they list. Salary aggregates by job therefore go through `v_jobs`. Aggregating by skill directly on the fact table is fine, because each job appears once per skill.
- **Industry uses a bridge table.** A job can belong to up to 3 industries. Putting industry on the fact table would multiply the rows again.
- **`dim_location`** parses LinkedIn's location strings (`"Denver, CO"`, `"Los Angeles, California, United States"`, `"Greater Boston"`, `"United States"`) into city, state, and `location_type`.
- **`dim_company`** has an `Unknown` member (`company_id = 0`) for the 497 postings without a company. `company_size` is LinkedIn's 0-7 size bucket.
- **`dim_time`** is keyed by `YYYYMMDD` and built from `original_listed_time`.

In [ ]:
print(wh.SCHEMA_SQL)

## Step 3 — Build and load the warehouse

This rebuilds `skillmap.db` from scratch. After loading, SQLite's `foreign_key_check` confirms that every fact row points to an existing dimension row.

In [ ]:
db_path = wh.build_warehouse(PROJECT_ROOT)
conn = sqlite3.connect(db_path)
wh.table_counts(conn)

## Step 4 — Dimension sanity checks

These show how many locations resolved to a state, and sample rows from each dimension. Jobs listed at metro or country level (e.g. *"United States"*, *"Greater Boston"*) have no state, so they drop out of the state-level query in Q3.

In [ ]:
display(pd.read_sql("SELECT location_type, COUNT(*) AS locations, SUM(state IS NULL) AS without_state "
                    "FROM dim_location GROUP BY location_type", conn))
for table in ["dim_location", "dim_company", "dim_time", "dim_skills"]:
    print(table)
    display(pd.read_sql(f"SELECT * FROM {table} ORDER BY RANDOM() LIMIT 3", conn))

## Step 5 — OLAP Query 1: top 10 most demanded skills

This counts jobs per extracted skill, using the fact table directly (each job appears once per skill). It also adds the average salary and the share of High-tier jobs, which gives a first look at which common skills pay more.

In [ ]:
title = list(wh.OLAP_QUERIES)[0]
print(wh.OLAP_QUERIES[title])
q1 = pd.read_sql(wh.OLAP_QUERIES[title], conn)
q1

## Step 6 — OLAP Query 2: average salary by experience level

This query runs on `v_jobs` (one row per job), so each salary is counted once. The tier percentages should match the Stage 2 cross-tab.

In [ ]:
title = list(wh.OLAP_QUERIES)[1]
print(wh.OLAP_QUERIES[title])
q2 = pd.read_sql(wh.OLAP_QUERIES[title], conn)
q2

## Step 7 — OLAP Query 3: job count by industry and location

This is a two-dimension roll-up: jobs are joined to industry through the bridge table and grouped by industry and state. A job with several industries counts once in each.

In [ ]:
title = list(wh.OLAP_QUERIES)[2]
print(wh.OLAP_QUERIES[title])
q3 = pd.read_sql(wh.OLAP_QUERIES[title], conn)
q3

## Step 8 — Save the summary

This writes table sizes, dimension checks, and all three query results to `outputs/03_summary.txt`.

In [ ]:
results = {title: pd.read_sql(sql, conn) for title, sql in wh.OLAP_QUERIES.items()}
summary = wh.build_summary(conn, results)
(OUTPUT_DIR / "03_summary.txt").write_text(summary, encoding="utf-8")
conn.close()
print(f"Saved {OUTPUT_DIR / '03_summary.txt'}")

## Next steps

- Stage 4 (association mining) can build transactions straight from the warehouse: `SELECT job_id, skill_name FROM job_postings JOIN dim_skills USING (skill_id) WHERE skill_type = 'extracted'`.
- Stage 5 (classification) can get `company_size` from `dim_company`, which `cleaned_jobs.csv` doesn't have. It's a required feature in the classification spec.